# 4 — Broadening the Scope: Multimodal Alignment and Fusion

**Cutting-EEG Workshop 2026 · UCSD · Deep Learning EEG Methods and Practice**
*Bonus notebook — the material most likely to be cut for time. It stands alone.*

EEG is rarely recorded alone. There's eye tracking, EMG, behavioral logs, sometimes
pupillometry. Most analyses throw the extra streams away, or analyze them separately.

Two problems stand between you and using them together, and the first is the one that
actually bites:

1. **Alignment** — different sampling rates, different clocks, different start times
2. **Fusion** — how to combine them in a network once aligned

### What we do here

1. Resample and align two streams onto a common time base
2. Quantify the cost of a timing offset — the bug that silently ruins multimodal studies
3. Build a **two-stream network** with a per-modality encoder
4. Compare **early / late / attention** fusion
5. Handle a missing modality at test time, because in real recordings a stream drops

### On the data

We use real EEG (BCI IV 2a) plus a **simulated gaze stream** derived from the trial
labels with controlled noise. Real synchronized EEG+gaze datasets exist but are large and
awkward to download in a tutorial.

Simulation is a deliberate choice with a real advantage: **we know the ground-truth
alignment**, so we can measure exactly what a 200 ms offset costs — impossible with real
data where the true offset is unknown. The alignment and fusion code is unchanged for
real data; only the loading differs.

---
**Runtime: `Runtime → Change runtime type → T4 GPU`.**

In [ ]:
%pip install -q "braindecode[moabb]"

import IPython
print("Install finished — restarting the runtime.")
print("This 'crash' notice is expected. Continue at Section 1 below.")
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import mne
import matplotlib.pyplot as plt
from scipy.signal import resample_poly

mne.set_log_level("ERROR")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260916
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"torch {torch.__version__} · device {DEVICE}")

## 1 · EEG, as before

In [ ]:
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import (
    Preprocessor, preprocess, exponential_moving_standardize,
    create_windows_from_events,
)
from torch.utils.data import DataLoader, TensorDataset

SUBJECT_ID = 3

dataset = MOABBDataset(dataset_name="BNCI2014_001", subject_ids=[SUBJECT_ID])
preprocess(dataset, [
    Preprocessor("pick_types", eeg=True, meg=False, eog=False),
    Preprocessor(lambda d: d * 1e6),
    Preprocessor("filter", l_freq=4.0, h_freq=38.0),
    Preprocessor(exponential_moving_standardize, factor_new=1e-3, init_block_size=1000),
], n_jobs=1)

EEG_SFREQ = dataset.datasets[0].raw.info["sfreq"]
windows = create_windows_from_events(
    dataset,
    trial_start_offset_samples=int(-0.5 * EEG_SFREQ),
    trial_stop_offset_samples=0,
    preload=True,
)

splits = windows.split("session")
train_w, test_w = splits["0train"], splits["1test"]


def to_arrays(ws):
    X = np.stack([ws[i][0] for i in range(len(ws))]).astype(np.float32)
    y = np.array([ws[i][1] for i in range(len(ws))], dtype=np.int64)
    return X, y


X_train_eeg, y_train = to_arrays(train_w)
X_test_eeg, y_test = to_arrays(test_w)

n_channels, n_times_eeg = X_train_eeg.shape[1], X_train_eeg.shape[2]
n_classes = len(windows.datasets[0].windows.event_id)
window_sec = n_times_eeg / EEG_SFREQ

print(f"EEG train {X_train_eeg.shape} · test {X_test_eeg.shape}")
print(f"{n_channels} channels @ {EEG_SFREQ} Hz · {window_sec:.1f}s windows · {n_classes} classes")

## 2 · A second modality at a different sampling rate

Real eye trackers run at 60, 120, 500, or 1000 Hz — rarely your EEG rate. We simulate a
**60 Hz gaze stream** with 3 channels (x position, y position, pupil diameter).

The simulation gives each motor imagery class a mildly distinct gaze signature, buried in
noise. `GAZE_SNR` controls how informative it is — set it to 0 and gaze becomes pure
noise, which is a useful sanity check on your fusion code (accuracy should drop to the
EEG-only level, not below).

In [ ]:
GAZE_SFREQ = 60.0
N_GAZE_CHANNELS = 3
GAZE_SNR = 0.6          # try 0.0 → gaze is pure noise


def simulate_gaze(labels, n_classes, duration_sec, sfreq, snr, seed=0):
    """Synthetic gaze: a weak class-dependent pattern plus pink-ish noise."""
    rng = np.random.RandomState(seed)
    n_trials = len(labels)
    n_samp = int(duration_sec * sfreq)
    t = np.linspace(0, duration_sec, n_samp)

    # Each class gets its own slow oscillation in x/y, and its own pupil ramp.
    signatures = []
    for c in range(n_classes):
        freq = 0.5 + 0.3 * c
        phase = c * np.pi / 3
        signatures.append(np.stack([
            np.sin(2 * np.pi * freq * t + phase),
            np.cos(2 * np.pi * freq * t + phase),
            np.tanh((t - duration_sec / 2) * (1 + 0.5 * c)),
        ]))

    out = np.zeros((n_trials, N_GAZE_CHANNELS, n_samp), dtype=np.float32)
    for i, lab in enumerate(labels):
        noise = rng.randn(N_GAZE_CHANNELS, n_samp)
        noise = np.cumsum(noise, axis=1) / np.sqrt(n_samp)   # drift, like real gaze
        noise = (noise - noise.mean(1, keepdims=True)) / (noise.std(1, keepdims=True) + 1e-8)
        out[i] = snr * signatures[lab] + (1 - snr) * noise

    return out


X_train_gaze = simulate_gaze(y_train, n_classes, window_sec, GAZE_SFREQ, GAZE_SNR, seed=1)
X_test_gaze = simulate_gaze(y_test, n_classes, window_sec, GAZE_SFREQ, GAZE_SNR, seed=2)

n_times_gaze = X_train_gaze.shape[2]
print(f"Gaze train {X_train_gaze.shape} · test {X_test_gaze.shape}")
print(f"\nEEG : {n_times_eeg} samples @ {EEG_SFREQ} Hz")
print(f"Gaze: {n_times_gaze} samples @ {GAZE_SFREQ} Hz")
print(f"\nSame {window_sec:.1f}s of time, different lengths. This is the alignment problem.")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)

t_eeg = np.arange(n_times_eeg) / EEG_SFREQ - 0.5
t_gaze = np.arange(n_times_gaze) / GAZE_SFREQ - 0.5

axes[0].plot(t_eeg, X_train_eeg[0, 7], lw=0.9, color="tab:blue")
axes[0].set_ylabel("EEG ch7 (µV)")
axes[0].set_title(f"Same trial, two modalities, two sampling rates")
axes[0].axvline(0, c="red", ls="--", label="cue"); axes[0].legend(); axes[0].grid(alpha=0.3)

for i, nm in enumerate(["gaze x", "gaze y", "pupil"]):
    axes[1].plot(t_gaze, X_train_gaze[0, i], lw=1.4, label=nm)
axes[1].axvline(0, c="red", ls="--")
axes[1].set_xlabel("time relative to cue (s)"); axes[1].set_ylabel("gaze (norm.)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 3 · Alignment

Two decisions.

**Which rate do you resample to?** Upsampling gaze 60→250 Hz invents detail that was
never measured. Downsampling EEG 250→60 Hz throws away beta and gamma — fatal for motor
imagery. We keep native rates and let each encoder produce a fixed-length embedding,
which is generally the right default. Resample only when a model demands identical
lengths.

`resample_poly` (polyphase) is the right tool when you must resample — it applies a
proper anti-aliasing filter. Naive slicing (`x[::4]`) aliases high frequencies down into
your band and is a silent corruption.

**How do you align the clocks?** In real recordings: shared trigger/TTL pulses, LSL
timestamps, or a common event. Never assume two devices started together — they didn't.

In [ ]:
def resample_to(x, sfreq_from, sfreq_to):
    """Polyphase resample along the last axis, with anti-aliasing."""
    from math import gcd
    up, down = int(sfreq_to), int(sfreq_from)
    g = gcd(up, down)
    return resample_poly(x, up // g, down // g, axis=-1).astype(np.float32)


gaze_up = resample_to(X_train_gaze[:2], GAZE_SFREQ, EEG_SFREQ)
print(f"Gaze upsampled to EEG rate : {X_train_gaze[:2].shape} -> {gaze_up.shape}")

eeg_down = resample_to(X_train_eeg[:2], EEG_SFREQ, GAZE_SFREQ)
print(f"EEG downsampled to gaze rate: {X_train_eeg[:2].shape} -> {eeg_down.shape}")

print("\nWe use NEITHER — each encoder handles its own rate. Shown so you have the tool.")

### How much does a timing offset cost?

The most damaging multimodal bug is a constant lag between streams — a trigger cable with
latency, a driver that timestamps on arrival rather than acquisition. Nothing crashes.
Your fusion model just quietly underperforms and you blame the architecture.

Because our gaze is simulated, we know the true alignment, so we can measure the damage
directly. We do this by cross-correlating the class-mean signals at various shifts.

In [ ]:
def alignment_score(eeg, gaze, labels, shift_samples, gaze_sfreq, eeg_sfreq):
    """Correlation between class-mean gaze and class-mean EEG envelope at a given shift."""
    gaze_at_eeg = resample_to(gaze, gaze_sfreq, eeg_sfreq)
    if shift_samples > 0:
        gaze_at_eeg = np.pad(gaze_at_eeg, ((0, 0), (0, 0), (shift_samples, 0)))[:, :, :-shift_samples or None]
    elif shift_samples < 0:
        s = -shift_samples
        gaze_at_eeg = np.pad(gaze_at_eeg, ((0, 0), (0, 0), (0, s)))[:, :, s:]

    eeg_env = np.abs(eeg).mean(axis=1)                # (trials, time)
    gaze_sig = gaze_at_eeg[:, 0, :]                   # gaze x

    corrs = []
    for c in np.unique(labels):
        m = labels == c
        a = eeg_env[m].mean(0); b = gaze_sig[m].mean(0)
        n = min(len(a), len(b))
        a, b = a[:n] - a[:n].mean(), b[:n] - b[:n].mean()
        denom = np.linalg.norm(a) * np.linalg.norm(b)
        corrs.append(abs(float(a @ b / denom)) if denom > 1e-8 else 0.0)
    return float(np.mean(corrs))


shifts_ms = np.arange(-500, 501, 50)
scores = [alignment_score(X_train_eeg, X_train_gaze, y_train,
                          int(ms / 1000 * EEG_SFREQ), GAZE_SFREQ, EEG_SFREQ)
          for ms in shifts_ms]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(shifts_ms, scores, "o-", lw=2)
best = shifts_ms[int(np.argmax(scores))]
ax.axvline(best, c="red", ls="--", label=f"peak at {best} ms")
ax.axvline(0, c="gray", ls=":", label="true alignment (0 ms)")
ax.set_xlabel("applied gaze shift (ms)"); ax.set_ylabel("EEG–gaze correspondence")
ax.set_title("Cross-correlation finds the offset between streams")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Peak at {best} ms (truth is 0 ms).")
print("\nOn real data, run this BEFORE modelling. A peak far from zero means your")
print("streams are misaligned — fix the timestamps, don't tune the network.")

## 4 · A two-stream network

The standard architecture: an encoder per modality, then a fusion step.

Separate encoders matter because the modalities are genuinely different. EEG is 22
channels at 250 Hz with oscillatory structure. Gaze is 3 channels at 60 Hz, smooth and
slow. One shared encoder would have to compromise; two specialized ones don't.

In [ ]:
class EEGBranch(nn.Module):
    """Compact EEGNet-style encoder → fixed-size embedding."""

    def __init__(self, n_channels, n_times, embed_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 40, 25, padding=12), nn.BatchNorm1d(40), nn.ELU(),
            nn.AvgPool1d(4), nn.Dropout(dropout),
            nn.Conv1d(40, 40, 15, padding=7, groups=4), nn.BatchNorm1d(40), nn.ELU(),
            nn.AvgPool1d(4), nn.Dropout(dropout),
            nn.Conv1d(40, embed_dim, 7, padding=3), nn.BatchNorm1d(embed_dim), nn.ELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.embed_dim = embed_dim

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GazeBranch(nn.Module):
    """Smaller encoder — gaze is lower-dimensional and much smoother."""

    def __init__(self, n_channels, n_times, embed_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_channels, 32, 9, padding=4), nn.BatchNorm1d(32), nn.ELU(),
            nn.AvgPool1d(2), nn.Dropout(dropout),
            nn.Conv1d(32, embed_dim, 5, padding=2), nn.BatchNorm1d(embed_dim), nn.ELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.embed_dim = embed_dim

    def forward(self, x):
        return self.net(x).squeeze(-1)


_e = EEGBranch(n_channels, n_times_eeg)
_g = GazeBranch(N_GAZE_CHANNELS, n_times_gaze)
print(f"EEG branch : {tuple(_e(torch.randn(4, n_channels, n_times_eeg)).shape)}")
print(f"Gaze branch: {tuple(_g(torch.randn(4, N_GAZE_CHANNELS, n_times_gaze)).shape)}")
print("\nDifferent input lengths, same embedding size — AdaptiveAvgPool1d does that,")
print("and it's why we never had to resample.")

## 5 · Three ways to fuse

| Strategy | Mechanism | Trade-off |
|---|---|---|
| **Early (concat)** | Concatenate embeddings, one MLP | Simple, strong baseline. Can let the dominant modality drown the other |
| **Late (decision)** | Classify each, average logits | Robust; handles a missing modality naturally. Can't model interactions |
| **Attention** | Each modality attends to the other | Learns *when* to trust which. More parameters, needs more data |

There is no universal winner. With one strong and one weak modality — our case, and the
common case with EEG+gaze — late fusion is often surprisingly hard to beat.

In [ ]:
class EarlyFusion(nn.Module):
    def __init__(self, eeg_branch, gaze_branch, n_classes, dropout=0.3):
        super().__init__()
        self.eeg, self.gaze = eeg_branch, gaze_branch
        d = eeg_branch.embed_dim + gaze_branch.embed_dim
        self.head = nn.Sequential(
            nn.Linear(d, 64), nn.ELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))

    def forward(self, x_eeg, x_gaze):
        return self.head(torch.cat([self.eeg(x_eeg), self.gaze(x_gaze)], dim=1))


class LateFusion(nn.Module):
    """Independent classifiers, averaged logits. `weight` sets the EEG/gaze balance."""

    def __init__(self, eeg_branch, gaze_branch, n_classes, weight=0.5):
        super().__init__()
        self.eeg, self.gaze = eeg_branch, gaze_branch
        self.eeg_head = nn.Linear(eeg_branch.embed_dim, n_classes)
        self.gaze_head = nn.Linear(gaze_branch.embed_dim, n_classes)
        self.weight = weight

    def forward(self, x_eeg, x_gaze, use_eeg=True, use_gaze=True):
        if use_eeg and use_gaze:
            return (self.weight * self.eeg_head(self.eeg(x_eeg))
                    + (1 - self.weight) * self.gaze_head(self.gaze(x_gaze)))
        if use_eeg:
            return self.eeg_head(self.eeg(x_eeg))
        return self.gaze_head(self.gaze(x_gaze))


class AttentionFusion(nn.Module):
    """Cross-attention between the two modality embeddings."""

    def __init__(self, eeg_branch, gaze_branch, n_classes, n_heads=4, dropout=0.3):
        super().__init__()
        self.eeg, self.gaze = eeg_branch, gaze_branch
        d = eeg_branch.embed_dim
        assert gaze_branch.embed_dim == d, "cross-attention needs matching embed dims"

        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(
            nn.Linear(d * 2, 64), nn.ELU(), nn.Dropout(dropout), nn.Linear(64, n_classes))

    def forward(self, x_eeg, x_gaze):
        e = self.eeg(x_eeg).unsqueeze(1)      # (B, 1, D)
        g = self.gaze(x_gaze).unsqueeze(1)
        tokens = torch.cat([e, g], dim=1)     # (B, 2, D)
        attended, _ = self.attn(tokens, tokens, tokens)
        tokens = self.norm(tokens + attended)  # residual
        return self.head(tokens.flatten(1))


class SingleModality(nn.Module):
    """Baseline: one modality only. Without these numbers, fusion results are unreadable."""

    def __init__(self, branch, n_classes):
        super().__init__()
        self.branch = branch
        self.head = nn.Linear(branch.embed_dim, n_classes)

    def forward(self, x):
        return self.head(self.branch(x))

In [ ]:
BATCH_SIZE = 64

train_ds = TensorDataset(torch.from_numpy(X_train_eeg),
                         torch.from_numpy(X_train_gaze),
                         torch.from_numpy(y_train))
test_ds = TensorDataset(torch.from_numpy(X_test_eeg),
                        torch.from_numpy(X_test_gaze),
                        torch.from_numpy(y_test))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Batches yield (eeg, gaze, label): "
      f"{[tuple(t.shape) for t in next(iter(train_loader))]}")

In [ ]:
from torch.optim import AdamW

N_EPOCHS = 60
LR = 1e-3


def train_model(model, name, modality="both", n_epochs=N_EPOCHS):
    model = model.to(DEVICE)
    opt = AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(xe, xg):
        if modality == "eeg":
            return model(xe)
        if modality == "gaze":
            return model(xg)
        return model(xe, xg)

    curve = []
    for _ in range(n_epochs):
        model.train()
        for xe, xg, y in train_loader:
            xe, xg, y = xe.to(DEVICE), xg.to(DEVICE), y.to(DEVICE)
            loss = crit(forward(xe, xg), y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval(); correct = total = 0
        with torch.no_grad():
            for xe, xg, y in test_loader:
                xe, xg, y = xe.to(DEVICE), xg.to(DEVICE), y.to(DEVICE)
                correct += (forward(xe, xg).argmax(1) == y).sum().item()
                total += len(y)
        curve.append(correct / total)

    print(f"{name:<24} best {max(curve):>6.1%} | final {curve[-1]:>6.1%}")
    return model, curve


def new_branches():
    return (EEGBranch(n_channels, n_times_eeg),
            GazeBranch(N_GAZE_CHANNELS, n_times_gaze))


print(f"{'Model':<24}{'best':>11}{'final':>9}")
print("-" * 44)

curves = {}
torch.manual_seed(SEED)
_, curves["EEG only"] = train_model(SingleModality(new_branches()[0], n_classes), "EEG only", "eeg")

torch.manual_seed(SEED)
_, curves["Gaze only"] = train_model(SingleModality(new_branches()[1], n_classes), "Gaze only", "gaze")

torch.manual_seed(SEED)
_, curves["Early fusion"] = train_model(EarlyFusion(*new_branches(), n_classes), "Early fusion")

torch.manual_seed(SEED)
late_model, curves["Late fusion"] = train_model(LateFusion(*new_branches(), n_classes), "Late fusion")

torch.manual_seed(SEED)
_, curves["Attention fusion"] = train_model(AttentionFusion(*new_branches(), n_classes), "Attention fusion")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

colors = {"EEG only": "tab:blue", "Gaze only": "tab:orange", "Early fusion": "tab:green",
          "Late fusion": "tab:red", "Attention fusion": "tab:purple"}
for name, curve in curves.items():
    style = "--" if "only" in name else "-"
    ax1.plot(range(1, len(curve) + 1), curve, style, lw=2, label=name, color=colors[name])
ax1.axhline(1/n_classes, ls=":", c="gray", label="chance")
ax1.set_xlabel("epoch"); ax1.set_ylabel("test accuracy")
ax1.set_title("Unimodal baselines vs fusion"); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

names = list(curves.keys())
best = [max(curves[n]) for n in names]
bars = ax2.barh(names, best, color=[colors[n] for n in names])
ax2.axvline(1/n_classes, ls=":", c="gray")
ax2.axvline(max(curves["EEG only"]), ls="--", c="tab:blue", label="EEG-only baseline")
ax2.set_xlabel("best test accuracy"); ax2.set_title("The bar to beat is EEG alone")
ax2.legend(fontsize=9)
for b, v in zip(bars, best):
    ax2.text(v + 0.008, b.get_y() + b.get_height()/2, f"{v:.1%}", va="center", fontsize=9)

plt.tight_layout(); plt.show()

eeg_only = max(curves["EEG only"])
print(f"EEG-only baseline: {eeg_only:.1%}\n")
for n in ["Early fusion", "Late fusion", "Attention fusion"]:
    print(f"{n:<20}{max(curves[n]):>7.1%}  ({max(curves[n])-eeg_only:+.1%} vs EEG alone)")

print("\nAlways report the unimodal baselines. A multimodal model that fails to beat")
print("its best single modality is a negative result — and a common unreported one.")

## 6 · When a stream drops out

Real recordings lose modalities. The eye tracker loses the pupil during a blink; a
subject wears mascara and calibration fails; an electrode comes loose.

**Late fusion handles this gracefully** — the branches are independent, so you can drop
one and still classify. Early and attention fusion cannot: their heads expect both
embeddings and produce garbage if you zero one.

Let's measure it.

In [ ]:
def eval_modes(model):
    model.eval()
    out = {}
    for label, kw in [("both", dict(use_eeg=True, use_gaze=True)),
                      ("EEG only", dict(use_eeg=True, use_gaze=False)),
                      ("gaze only", dict(use_eeg=False, use_gaze=True))]:
        correct = total = 0
        with torch.no_grad():
            for xe, xg, y in test_loader:
                xe, xg, y = xe.to(DEVICE), xg.to(DEVICE), y.to(DEVICE)
                correct += (model(xe, xg, **kw).argmax(1) == y).sum().item()
                total += len(y)
        out[label] = correct / total
    return out


modes = eval_modes(late_model)
print("Late-fusion model, evaluated with modalities dropped at TEST time:")
print("(the model was trained on both; nothing was retrained)\n")
for k, v in modes.items():
    print(f"  {k:<12}{v:>7.1%}")

print(f"\nDropping gaze costs {modes['both']-modes['EEG only']:+.1%}")
print(f"Dropping EEG  costs {modes['both']-modes['gaze only']:+.1%}")
print("\nThe model degrades instead of failing. For clinical or BCI deployment,")
print("that robustness usually matters more than a point of peak accuracy.")

### Modality dropout during training

You can make this robustness explicit: randomly drop a modality during training, so the
model never learns to depend on both being present. Same idea as regular dropout, applied
at the stream level.

In [ ]:
class LateFusionWithDropout(LateFusion):
    """Late fusion + random modality dropout during training."""

    def __init__(self, *a, p_drop=0.25, **kw):
        super().__init__(*a, **kw)
        self.p_drop = p_drop

    def forward(self, x_eeg, x_gaze, use_eeg=True, use_gaze=True):
        if self.training and torch.rand(1).item() < self.p_drop:
            if torch.rand(1).item() < 0.5:
                use_gaze = False
            else:
                use_eeg = False
        return super().forward(x_eeg, x_gaze, use_eeg, use_gaze)


torch.manual_seed(SEED)
robust_model, curve_robust = train_model(
    LateFusionWithDropout(*new_branches(), n_classes, p_drop=0.25), "Late + mod-dropout")

modes_robust = eval_modes(robust_model)
print(f"\n{'mode':<12}{'standard':>11}{'+ dropout':>12}")
print("-" * 35)
for k in modes:
    print(f"{k:<12}{modes[k]:>10.1%}{modes_robust[k]:>12.1%}")

print("\nModality dropout usually trades a little peak accuracy for better")
print("degraded-mode performance. Whether that's worth it depends on deployment.")

## 7 · Using your own second modality

The template. Replace the simulation with your real stream; everything else stands.

In [ ]:
TEMPLATE = '''
# ---- Multimodal EEG template -------------------------------------------
# 1. ALIGN FIRST — before any modelling
#    · Find a common clock: TTL triggers, LSL timestamps, a shared event
#    · Verify with cross-correlation. A peak far from 0 = misalignment.
#    · Never assume two devices started simultaneously.

# 2. EPOCH BOTH STREAMS to the same events and the same time window.
#    Keep native sampling rates; AdaptiveAvgPool1d absorbs the length difference.
X_eeg   = ...   # (n_trials, n_eeg_channels,  n_eeg_samples)
X_other = ...   # (n_trials, n_other_channels, n_other_samples)
y       = ...   # (n_trials,)

# 3. ONE ENCODER PER MODALITY — they have different statistics
eeg_branch   = EEGBranch(n_eeg_channels,   n_eeg_samples,   embed_dim=64)
other_branch = GazeBranch(n_other_channels, n_other_samples, embed_dim=64)

# 4. FUSE. Start with late fusion: strong baseline, robust to dropout.
model = LateFusion(eeg_branch, other_branch, n_classes)

# 5. ALWAYS TRAIN THE UNIMODAL BASELINES.
#    If fusion doesn't beat the best single modality, it isn't working.
# ------------------------------------------------------------------------
'''
print(TEMPLATE)

## Recap

**Alignment is the hard part; fusion is the easy part.** Most multimodal EEG failures are
timing bugs, not architecture bugs. Verify alignment by cross-correlation before you write
a single line of model code.

Other things worth carrying:

- **Keep native sampling rates.** Adaptive pooling makes resampling unnecessary, and
  resampling always costs you something.
- **Late fusion is a strong default** — simple, robust, degrades gracefully.
- **Unimodal baselines are mandatory.** Fusion that doesn't beat its best single modality
  is a negative result.
- **Plan for dropout.** Streams fail in real recordings. Decide now what the model does
  when one goes missing.

### Try it yourself

1. Set `GAZE_SNR = 0.0` so gaze is pure noise. Fusion should match — not beat — EEG
   alone. If it beats it, you have a leak.
2. Set `GAZE_SNR = 0.9`. When gaze dominates, does attention fusion learn to lean on it?
3. Introduce a real misalignment: roll the gaze array by 50 samples before training. How
   much accuracy does a 200 ms offset cost?
4. Add a third modality (e.g. simulated EMG). The two-stream pattern extends directly.

---

## End of the series

| Notebook | Idea |
|---|---|
| 1 · Core pipeline | Raw → Preprocess → Window → Dataset → Model |
| 2 · Transformers | Tokenization, attention, and why small data limits it |
| 3 · Foundation models | Pretrain on unlabeled EEG, transfer to your study |
| 4 · Multimodal | Align first, then fuse |

The thread running through all four: **the pipeline shape is constant.** Swap the model,
swap the pretraining, add a modality — the `Raw → Window → Dataset → Model` chain from
Notebook 1 never changes. That's what makes this stuff reusable in a real lab.